In [7]:
import pandas as pd
import numpy as np
from plots import plot_appendix, plot_main_paper
from kcit import efficient_kci_test
from scipy.special import expit # expit applies sigmoid to a Series
import pickle

# muskaan runs this inside kcit_sim_env


In [8]:
RESULTS_DIR = 'kcit_randomized_graph_medicare_data_results'

# Kcit Experiments

In [13]:
def generate_sgc_datasets(n,
                          true_upcoding_rates: list[float], true_downcoding_rates: list[float],
                          genuine_modifications_a_1_on_xstar: list[float], genuine_modifications_a_0_on_xstar: list[float]):
    """
    n = number of features (X*'s)
    assuming just 1 downstream variable Y for now, I believe this is WLOG
    all other args are lists of values, 1 list entry per feature (X*) """

    # Load dataset
    df = pd.read_pickle('datasets/medicare_data.pkl')
    df = df.sample(frac=0.005).reset_index(drop=True)
    df = df.dropna()
    print(len(df))

    df = df.rename(columns={
        'AGE_AT_END_REF_YR_18': 'AGE',
        'SEX_IDENT_CD_18': 'SEX',
        'BENE_RACE_CD_18_1': 'RACE'
    })

    # Drop columns that are not needed
    df = df[['SEX', 'RACE', 'AGE']]
    # Min-max scale the data
    df['AGE'] = (df['AGE'] - df['AGE'].min()) / (df['AGE'].max() - df['AGE'].min())
    
    # Use sex as selection bias
    agent_prob = 0.40  + (1-df['SEX'])*0.20
    df['AGENT'] = np.random.binomial(1, agent_prob, len(df))

    causal_effect_of_x_on_y_list = []
    y_prob = 0.05 \
                + df['RACE']*0.2 \
                + df['SEX']*0.25 \
                + np.square(df['AGE'])*0.1
    

    for i in range(n):
        variable_name = f"X{i}"
        
        x_prob = 0.15 \
                        + df['RACE']*df['SEX']*0.2 \
                        + np.square(df['AGE'])*0.1 \
                        + df['AGENT'] * genuine_modifications_a_1_on_xstar[i] + (1-df['AGENT']) * genuine_modifications_a_0_on_xstar[i]
        df[variable_name] = np.random.binomial(1, expit(x_prob), len(df))

        # Strategically misreport the dataset (misreported employment status)
        prob_required_for_upcoding_rate = ((x_prob / (1-true_upcoding_rates[i])) - x_prob) / (1 - x_prob)
        prob_required_for_downcoding_rate = (((1 - x_prob) / (1-true_downcoding_rates[i])) - (1 - x_prob)) / x_prob

        df[variable_name] = (df[variable_name] + df['AGENT']*(1-df[variable_name]) * np.random.binomial(1, expit(prob_required_for_upcoding_rate),  len(df))
                            - df[variable_name]*(1-df['AGENT']) * np.random.binomial(1, expit(prob_required_for_downcoding_rate), len(df)))
        
        values = np.array([0.0, 0.0, 0.0, 0.0, 0.90, 1.0, 1.1, 1.2]) # we have to take large values here because this is ultimately going through a sigmoid??
        causal_effect = np.random.choice(values)
        causal_effect_of_x_on_y_list.append(causal_effect)
        y_prob += df[variable_name] * causal_effect
        
    # muskaan comment: te i.e., x1_causal_effect_on_y, is beta(x) in the paper
    df['Y'] = np.random.binomial(1, expit(y_prob), len(df)) # expit applies sigmoid to a Series

    return df, causal_effect_of_x_on_y_list

# Kcit

In [14]:
num_sims = 10
n = 10 # n = number of features (HCCs) in this simulation

# Dataframes to keep track of results
rows = []

num_correct = 0

# Perform a few simulations for each method
for sim in range(num_sims):
    # Set the random seed
    np.random.seed(sim)

    # Generate dataset for simulation
    df, causal_effects_of_x_on_y = generate_sgc_datasets(n, [0.1, 0.2, 0.3, 0.0, 0.05, 0.06, 0.07, 0.04, 0.03, 0.3, 0.4, 0.3, 0.1],
                    [0.04, 0.1, 0.2, 0.3, 0.0, 0.05, 0.06, 0.4, 0.03, 0.6, 0.1, 0.4, 0.3, 0.1],  
                    [ 0.04, 0.03, 0.6, 0.1, 0.2, 0.3, 0.0, 0.05, 0.01, 0.07, 0.4, 0.3, 0.1],
                    [ 0.3, 0.05, 0.06, 0.17, 0.03, 0.2, 0.1, 0.2, 0.3, 0.0, 0.4, 0.3, 0.1])

    x_list = []
    for i in range(n):
        if causal_effects_of_x_on_y[i] == 0:
            x_list.append(f"X{i}")

    if len(x_list) == 0:
        continue

    # now x_list contains all the features that SHOULD be conditionally independent of Y given confounders

    p_value = efficient_kci_test(df, x_list, ['AGENT', 'SEX', 'RACE', 'AGE'])

    rows.append({
        "simulation_number": sim,
        "p_value": p_value,
        "accurate": (p_value >= 0.05),
    })

df = pd.DataFrame(rows, columns=["simulation_number", "p_value", "accurate"])


with open(f'{RESULTS_DIR}/df.pkl', 'wb') as f:
    pickle.dump(df, f)

5640
5640
5640
5640
5640
5640
5640
5640
5640
5640


In [15]:
with open(f'{RESULTS_DIR}/df.pkl', 'rb') as f:
    loaded_df = pickle.load(f)

In [16]:
print(loaded_df)


   simulation_number   p_value  accurate
0                  0  0.654406      True
1                  1  0.402334      True
2                  2  0.522065      True
3                  3  0.653333      True
4                  4  0.243428      True
5                  5  0.414188      True
6                  6  0.673280      True
7                  7  0.239389      True
8                  8  0.443376      True
9                  9  0.225740      True
